# Train the multi-part tomato models (Google Colab, GPU runtime)

Creates `part_model.keras` (which plant part is in the photo) and one disease model per part.

## 1. Prepare your data in Google Drive
```
MyDrive/tomato_data/
  diseases/
    fruit/   fruit_healthy/  fruit_blossom_end_rot/  fruit_anthracnose/  fruit_late_blight/ ...
    stem/    stem_healthy/   stem_bacterial_canker/ ...
    flower/  flower_healthy/ flower_gray_mold/ ...
    root/    root_healthy/   root_knot_nematode/ root_rot/
    leaf/    (optional: any leaf images, only used to teach the part classifier)
  parts/
    other/   (200+ photos that are NOT tomato parts: other plants, soil, hands, tools, objects)
```
**Folder names are the class keys** and must match the names in `remedies.py` (e.g. `fruit_blossom_end_rot`), otherwise the site shows generic advice.

## 2. Where to get images (checked online; availability varies)
- **Fruit:** TOM2024 on Hugging Face (`Project-AgML/TOM2024_disease_classification`: abiotic disorder, blossom end rot, healthy fruit, late blight); Mendeley Data "Multiclass Image Dataset of Tomato Pathologies" (healthy / infected fruit); Mendeley TFDD `ktfnhjspjn` (anthracnose, blossom end rot, cracking, early blight, fruitworm, mold, healthy, but in bounding-box format, so crop the boxes into class folders).
- **Stem / flower / roots:** no ready-made public classification datasets were found. Options: collect and label your own photos (aim for 150+ per class), ask a local agriculture university/extension office, or crop stem regions from the Mendeley "common tomato diseases" dataset (viral, gray mold, bacterial wilt, boxes cover leaves, fruits and stems).
- Check each dataset's licence before using it.

A part only works on the website once its model exists, so train only the parts you have data for.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/tomato_data'
OUT = '/content/model_out'
import os; os.makedirs(OUT, exist_ok=True)
TRAIN_LEAF = False   # keep False: your existing leaf model (tomato_cnn.keras) is already good

In [ ]:
import json, random, shutil, glob
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

IMG_SIZE, BATCH, AUTOTUNE = 160, 32, tf.data.AUTOTUNE

def train_folder_model(data_dir, name, epochs=10, ft_epochs=8):
    kw = dict(validation_split=0.15, seed=42, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH)
    train_ds = tf.keras.utils.image_dataset_from_directory(data_dir, subset='training', **kw)
    val_ds = tf.keras.utils.image_dataset_from_directory(data_dir, subset='validation', **kw)
    class_names = train_ds.class_names
    counts = {c: len(os.listdir(os.path.join(data_dir, c))) for c in class_names}
    total = sum(counts.values())
    class_weight = {i: total / (len(class_names) * counts[c]) for i, c in enumerate(class_names)}  # handles imbalance
    print(name, counts)
    train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
    val_ds = val_ds.cache().prefetch(AUTOTUNE)

    aug = tf.keras.Sequential([layers.RandomFlip('horizontal_and_vertical'), layers.RandomRotation(0.15),
                               layers.RandomZoom(0.2), layers.RandomContrast(0.25), layers.RandomBrightness(0.2)])
    base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
    base.trainable = False
    inp = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = preprocess_input(aug(inp))
    x = layers.GlobalAveragePooling2D()(base(x, training=False))
    out = layers.Dense(len(class_names), activation='softmax')(layers.Dropout(0.3)(x))
    model = tf.keras.Model(inp, out)

    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(train_ds, validation_data=val_ds, epochs=epochs, class_weight=class_weight)
    base.trainable = True
    for l in base.layers[:-40]: l.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(train_ds, validation_data=val_ds, epochs=ft_epochs, class_weight=class_weight)
    print('VALIDATION:', model.evaluate(val_ds))

    model.save(f'{OUT}/{name}_model.keras')
    with open(f'{OUT}/{name}_classes.json', 'w') as f:
        json.dump({'img_size': IMG_SIZE, 'keys_by_index': class_names}, f, indent=2)
    print('Saved', name, class_names)

In [ ]:
# Disease model for every part folder you filled in
for part in sorted(os.listdir(f'{DATA}/diseases')):
    if part == 'leaf' and not TRAIN_LEAF: continue
    train_folder_model(f'{DATA}/diseases/{part}', part)

In [ ]:
# Build the part-classifier dataset: sample images from each disease folder, plus your 'other' folder
PARTS_DIR = f'{DATA}/parts'
for part in os.listdir(f'{DATA}/diseases'):
    files = glob.glob(f'{DATA}/diseases/{part}/*/*')
    os.makedirs(f'{PARTS_DIR}/{part}', exist_ok=True)
    for i, f in enumerate(random.sample(files, min(800, len(files)))):
        shutil.copy(f, f'{PARTS_DIR}/{part}/{i}_{os.path.basename(f)}')
print(os.listdir(PARTS_DIR))   # expect: leaf, fruit, stem, flower, root, other (only the ones you have)

In [ ]:
train_folder_model(PARTS_DIR, 'part')

In [ ]:
# Zip everything and download. Copy the files into the website's model/ folder.
shutil.make_archive('/content/trained_models', 'zip', OUT)
from google.colab import files
files.download('/content/trained_models.zip')